# AI Detectability Experiment

Main experiment notebook. Run cells sequentially from top to bottom.

**Prerequisites:** See `INSTRUCTIONS.md` for environment setup and manual data-collection steps (Steps 0-2).

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore", message="coroutine 'Kernel.shell_main' was never awaited")

from ai_text_quality.config import load_all_tasks, load_style_rules
from ai_text_quality.generate import build_all_prompts, build_humanize_prompts, load_all_generated, MODELS, WORD_TARGETS
from ai_text_quality.io_utils import read_jsonl, write_jsonl, read_markdown
from ai_text_quality.detect import create_detection_template, load_detection_from_csv
from ai_text_quality.factcheck import score_document
from ai_text_quality.linguistic import extract_features, compute_style_distance
from ai_text_quality.models import GeneratedText, LinguisticFeatures
from ai_text_quality.paths import (
    GENERATED_DIR, HUMAN_BASELINES_DIR,
    DETECTION_DIR, FACTUAL_DIR, LINGUISTIC_DIR, SUMMARY_DIR,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon, kruskal, spearmanr
from pathlib import Path

print("Imports OK")

Imports OK


## Helper: collect all texts (generated + human baselines)

In [2]:
def collect_all_texts() -> list[GeneratedText]:
    """Load every generated text and every human baseline into a flat list."""
    tasks = load_all_tasks()
    texts: list[GeneratedText] = load_all_generated(
        tasks,
        runs=2,
        model_keys=["claude", "codex"],
        word_targets=["800-1000", "1500-2000"],
    )

    # Human baselines
    if HUMAN_BASELINES_DIR.exists():
        for project_dir in sorted(HUMAN_BASELINES_DIR.iterdir()):
            if not project_dir.is_dir():
                continue
            for md_file in sorted(project_dir.glob("*.md")):
                content = read_markdown(md_file)
                if content.startswith("---"):
                    parts = content.split("---", 2)
                    content = parts[2].strip() if len(parts) >= 3 else content
                texts.append(GeneratedText(
                    task_id=md_file.stem,
                    condition="human_baseline",
                    run_id="run_01",
                    text=content,
                    model="human",
                    timestamp="",
                    token_usage={},
                ))

    print(f"Collected {len(texts)} texts")
    return texts

---
## Step 3a: Build C1 + C2 Prompts

Builds all prompts for conditions C1 (context_rich) and C2 (style_constrained) and saves them to `prompts.jsonl`.

After running this cell, paste each prompt into the appropriate CLI tool (Claude Code, Codex CLI, or Gemini CLI). Each prompt includes a save instruction so the CLI tool writes the output directly to disk.

In [ ]:
tasks = load_all_tasks()
style_rules = load_style_rules()
print(f"Loaded {len(tasks)} tasks across {len(set(t.project for t in tasks))} projects")

# Build C1 + C2 prompts
prompts = build_all_prompts(
    tasks,
    style_rules,
    runs=2,
    model_keys=["claude"],
    word_targets=["800-1000", "1500-2000"],
)

# Show first prompt as a sample
print(f"\n--- Sample prompt (paste into CLI tool) ---")
print(prompts[0]["prompt"][:300] + "...")

---
## Step 3b: Build C3 (Humanize) Prompts

Reads C1 output files from disk and builds C3 rewrite prompts. **Run this only after all C1 outputs have been generated.**

Then paste each C3 prompt into the CLI tool.

In [3]:
tasks = load_all_tasks()

humanize_prompts = build_humanize_prompts(
    tasks,
    runs=2,
    model_keys=["claude"],
    word_targets=["800-1000", "1500-2000"],
)

Built 48 C3 prompts → 2 files in data/generated/prompts/
  prompts_claude_humanized_long.txt
  prompts_claude_humanized_medium.txt


---
## Step 3c: Load Generated Outputs

Reads all output files from disk and saves metadata. **Run this after all C1, C2, and C3 outputs have been generated.**

In [ ]:
tasks = load_all_tasks()

results = load_all_generated(
    tasks,
    runs=2,
    model_keys=["claude"],
    word_targets=["800-1000", "1500-2000"],
)

print(f"Loaded {len(results)} texts")

---
## Step 4: Run AI Detection

Creates a CSV template with one row per text. Fill in `gptzero_generated_prob` and `originality_ai_score` columns (values 0-1) after pasting each text into the respective web UIs, then re-run the load cell to import results.

**Workflow:**
1. Run cell 4a to generate `detection_template.csv`
2. Open the CSV, paste each text into GPTZero / Originality.ai, fill in scores
3. Run cell 4b to load the filled CSV into `detection_results.jsonl`

In [4]:
# --- 4a: Create template CSV ---
from ai_text_quality.detect import create_detection_template, load_detection_from_csv

all_texts = collect_all_texts()

template_path = DETECTION_DIR / "detection_template.csv"
create_detection_template(all_texts, template_path)
print(f"\nOpen {template_path} and fill in the score columns.")
print("Then run the next cell to load results.")

Collected 310 texts
Wrote detection template with 310 rows → /Users/yongkyunlee/ai-detectability-research/code/data/results/detection/detection_template.csv

Open /Users/yongkyunlee/ai-detectability-research/code/data/results/detection/detection_template.csv and fill in the score columns.
Then run the next cell to load results.


In [5]:
# --- 4b: Load filled CSV ---
template_path = DETECTION_DIR / "detection_template.csv"
detection_results = load_detection_from_csv(template_path)

DETECTION_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(
    DETECTION_DIR / "detection_results.jsonl",
    [r.model_dump() for r in detection_results],
)
print(f"Saved {len(detection_results)} detection results")

Skipped 310 rows with no scores filled in
Loaded 0 detection results from /Users/yongkyunlee/ai-detectability-research/code/data/results/detection/detection_template.csv
Saved 0 detection results


---
## Step 5a: Build Fact-Check Prompts

Builds a combined extract+verify prompt for each generated text across all models (Claude, Codex). Each prompt tells the CLI tool to:
1. Read the generated blog post
2. Read the reference documentation from the context directory
3. Extract claims and verify them against the docs
4. Save the results to a file

After running this cell, paste each prompt into a CLI tool (use a **different** model than the generator).

In [ ]:
from ai_text_quality.factcheck import build_factcheck_prompts

tasks = load_all_tasks()

MODEL_KEYS = ["claude", "codex"]

all_factcheck_prompts = []
for mk in MODEL_KEYS:
    gen_texts = load_all_generated(
        tasks,
        runs=2,
        model_keys=[mk],
        word_targets=["800-1000", "1500-2000"],
    )
    if gen_texts:
        prompts = build_factcheck_prompts(tasks, gen_texts, model_key=mk)
        all_factcheck_prompts.extend(prompts)

print(f"\nTotal: {len(all_factcheck_prompts)} fact-check prompts across {MODEL_KEYS}")

---
## Step 5b: Load Fact-Check Results

Run this after all fact-check output files have been saved by the CLI tool. Loads results for all models (Claude, Codex) and saves `factcheck_results.jsonl`.

In [3]:
from ai_text_quality.factcheck import load_factcheck_outputs

tasks = load_all_tasks()

MODEL_KEYS = ["claude", "codex"]

all_fact_results = []
for mk in MODEL_KEYS:
    gen_texts = load_all_generated(
        tasks,
        runs=2,
        model_keys=[mk],
        word_targets=["800-1000", "1500-2000"],
    )
    if gen_texts:
        results = load_factcheck_outputs(tasks, gen_texts, model_key=mk)
        all_fact_results.extend(results)

FACTUAL_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(
    FACTUAL_DIR / "factcheck_results.jsonl",
    [
        {
            "task_id": r.task_id,
            "condition": r.condition,
            "run_id": r.run_id,
            "model": r.model,
            "word_target": r.word_target,
            "factual_precision": r.factual_precision,
            "total_claims": r.total_claims,
            "incorrect_claims": r.incorrect_claims,
        }
        for r in all_fact_results
    ],
)

print(f"\nSaved {len(all_fact_results)} fact-check results\n")
for r in all_fact_results:
    print(f"  {r.condition}/{r.task_id}/{r.run_id}: "
          f"model={r.model} "
          f"precision={r.factual_precision:.2f} "
          f"claims={r.total_claims} "
          f"incorrect={r.incorrect_claims}")

Loaded 144 generated texts
Loaded 36 fact-check results from output files
Loaded 24 generated texts
Loaded 24 fact-check results from output files

Saved 60 fact-check results

  context_rich/crewai_001/run_01: model=claude-opus-4-6 (Claude Code) precision=0.90 claims=32 incorrect=3
  style_constrained/crewai_001/run_01: model=claude-opus-4-6 (Claude Code) precision=1.00 claims=48 incorrect=0
  humanized/crewai_001/run_01: model=claude-opus-4-6 (Claude Code) precision=0.94 claims=33 incorrect=2
  context_rich/crewai_004/run_01: model=claude-opus-4-6 (Claude Code) precision=0.98 claims=46 incorrect=1
  style_constrained/crewai_004/run_01: model=claude-opus-4-6 (Claude Code) precision=1.00 claims=57 incorrect=0
  humanized/crewai_004/run_01: model=claude-opus-4-6 (Claude Code) precision=0.97 claims=41 incorrect=1
  context_rich/crewai_006/run_01: model=claude-opus-4-6 (Claude Code) precision=0.98 claims=63 incorrect=1
  style_constrained/crewai_006/run_01: model=claude-opus-4-6 (Claude C

---
## Step 6: Linguistic Features

Fully automatic — no manual interaction needed.

In [3]:
all_texts = collect_all_texts()

ling_results = []
for gen in all_texts:
    features = extract_features(
        gen.text, gen.task_id, gen.condition, gen.run_id,
        model=gen.model, word_target=getattr(gen, "word_target", ""),
    )
    ling_results.append(features)
    print(f"{gen.condition}/{gen.task_id}: "
          f"discourse_markers={features.discourse_marker_rate:.2f} "
          f"contractions={features.contraction_rate:.2f}")

LINGUISTIC_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(
    LINGUISTIC_DIR / "linguistic_features.jsonl",
    [r.model_dump() for r in ling_results],
)
print(f"\nSaved {len(ling_results)} feature records")

Loaded 168 generated texts
Collected 190 texts
context_rich/crewai_001: discourse_markers=0.00 contractions=0.00
style_constrained/crewai_001: discourse_markers=0.00 contractions=1.00
humanized/crewai_001: discourse_markers=0.00 contractions=1.00
context_rich/crewai_001: discourse_markers=0.00 contractions=1.00
style_constrained/crewai_001: discourse_markers=0.00 contractions=1.00
humanized/crewai_001: discourse_markers=0.00 contractions=1.00
context_rich/crewai_004: discourse_markers=0.04 contractions=0.00
style_constrained/crewai_004: discourse_markers=0.00 contractions=0.82
humanized/crewai_004: discourse_markers=0.00 contractions=1.00
context_rich/crewai_004: discourse_markers=0.00 contractions=0.00
style_constrained/crewai_004: discourse_markers=0.00 contractions=1.00
humanized/crewai_004: discourse_markers=0.00 contractions=1.00
context_rich/crewai_006: discourse_markers=0.00 contractions=0.00
style_constrained/crewai_006: discourse_markers=0.00 contractions=0.91
humanized/crewai

---
## Step 7: Analysis

Summary tables, style distance, and hypothesis tests.

In [ ]:
# Load all results
detection = pd.DataFrame(read_jsonl(DETECTION_DIR / "detection_results.jsonl"))
factual = pd.DataFrame(read_jsonl(FACTUAL_DIR / "factcheck_results.jsonl"))
linguistic = pd.DataFrame(read_jsonl(LINGUISTIC_DIR / "linguistic_features.jsonl"))

FEATURE_COLS = [
    "sent_len_std", "sent_len_mean", "vocab_diversity",
    "contraction_rate", "first_person_rate", "discourse_marker_rate",
    "list_density", "passive_ratio", "paragraph_len_std", "specificity_score",
]
print(f"Loaded: {len(detection)} detection, {len(factual)} factual, {len(linguistic)} linguistic")

### 7a. Summary tables

In [ ]:
det_summary = detection.groupby("condition").agg(
    gptzero_human_prob_mean=("gptzero_human_prob", "mean"),
    gptzero_human_prob_std=("gptzero_human_prob", "std"),
    originality_human_mean=("originality_human_score", "mean"),
    originality_human_std=("originality_human_score", "std"),
).round(3)
det_summary

In [ ]:
fact_summary = factual.groupby("condition").agg(
    precision_mean=("factual_precision", "mean"),
    precision_std=("factual_precision", "std"),
    total_claims_mean=("total_claims", "mean"),
    incorrect_mean=("incorrect_claims", "mean"),
).round(3)
fact_summary

In [ ]:
ling_summary = linguistic.groupby("condition")[FEATURE_COLS].mean().round(3)
ling_summary

### 7b. Style distance from human baseline

In [ ]:
human_rows = linguistic[linguistic["condition"] == "human_baseline"]
baseline_mean = human_rows[FEATURE_COLS].mean().to_dict()

distances = []
for _, row in linguistic.iterrows():
    features = LinguisticFeatures(**row.to_dict())
    dist = compute_style_distance(features, baseline_mean)
    distances.append({
        "task_id": row["task_id"],
        "condition": row["condition"],
        "run_id": row["run_id"],
        "style_distance": dist,
    })
dist_df = pd.DataFrame(distances)
dist_df.groupby("condition")["style_distance"].agg(["mean", "std"]).round(3)

### 7c. Hypothesis tests

In [ ]:
det_task_means = detection.groupby(["task_id", "condition"])["gptzero_human_prob"].mean().reset_index()

def paired_wilcoxon(cond_a: str, cond_b: str) -> dict:
    """Paired Wilcoxon test: is cond_b > cond_a on gptzero_human_prob?"""
    a = det_task_means[det_task_means["condition"] == cond_a].set_index("task_id")["gptzero_human_prob"]
    b = det_task_means[det_task_means["condition"] == cond_b].set_index("task_id")["gptzero_human_prob"]
    common = a.index.intersection(b.index)
    if len(common) < 5:
        return {"n": len(common), "note": "too few pairs"}
    stat, p = wilcoxon(b.loc[common], a.loc[common], alternative="greater")
    effect_size = (b.loc[common] - a.loc[common]).mean()
    return {"n": len(common), "statistic": stat, "p_value": p, "mean_diff": effect_size}

print("H1 (C2 > C1):", paired_wilcoxon("context_rich", "style_constrained"))
print("H2 (C3 > C1):", paired_wilcoxon("context_rich", "humanized"))

c1_mean = det_task_means[det_task_means["condition"] == "context_rich"]["gptzero_human_prob"].mean()
c2_mean = det_task_means[det_task_means["condition"] == "style_constrained"]["gptzero_human_prob"].mean()
c5_mean = det_task_means[det_task_means["condition"] == "human_baseline"]["gptzero_human_prob"].mean()
if c5_mean != c1_mean:
    gap_closed = (c2_mean - c1_mean) / (c5_mean - c1_mean)
    print(f"\nH3: C2 closes {gap_closed:.1%} of the C1-to-human gap")

# Note: Apply Bonferroni correction — multiply p-values by 2

---
## Step 8: Figures

In [ ]:
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
ORDER = ["context_rich", "style_constrained", "humanized", "human_baseline"]

### Figure 1: GPTZero human_prob by condition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=detection, x="condition", y="gptzero_human_prob", order=ORDER, ax=ax, ci=95)
ax.set_ylabel("GPTZero Human Probability")
ax.set_xlabel("Condition")
ax.set_title("AI Detectability by Condition")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(SUMMARY_DIR / "fig1_gptzero_by_condition.png", dpi=150)
plt.show()

### Figure 2: Originality.ai human score by condition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=detection, x="condition", y="originality_human_score", order=ORDER, ax=ax, ci=95)
ax.set_ylabel("Originality.ai Human Score")
ax.set_xlabel("Condition")
ax.set_title("AI Detectability (Originality.ai) by Condition")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(SUMMARY_DIR / "fig2_originality_by_condition.png", dpi=150)
plt.show()

### Figure 3: Pareto scatter (detectability vs accuracy)

In [ ]:
merged = detection.merge(factual, on=["task_id", "condition", "run_id"], how="inner")
fig, ax = plt.subplots(figsize=(8, 6))
palette = {
    "context_rich": "#3498db",
    "style_constrained": "#2ecc71", "humanized": "#9b59b6",
}
for cond, color in palette.items():
    subset = merged[merged["condition"] == cond]
    ax.scatter(subset["gptzero_human_prob"], subset["factual_precision"],
               label=cond, color=color, alpha=0.6, s=40)
ax.set_xlabel("GPTZero Human Probability (less detectable ->)")
ax.set_ylabel("Factual Precision")
ax.set_title("Detectability vs Factual Accuracy Trade-off")
ax.legend()
plt.tight_layout()
plt.savefig(SUMMARY_DIR / "fig3_pareto_scatter.png", dpi=150)
plt.show()

### Figure 4: Radar chart of linguistic features

In [ ]:
ling_means = linguistic.groupby("condition")[FEATURE_COLS].mean()
ling_norm = (ling_means - ling_means.min()) / (ling_means.max() - ling_means.min() + 1e-9)

angles = np.linspace(0, 2 * np.pi, len(FEATURE_COLS), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for cond in ORDER:
    if cond not in ling_norm.index:
        continue
    values = ling_norm.loc[cond].tolist()
    values += values[:1]
    ax.plot(angles, values, label=cond, linewidth=2)
    ax.fill(angles, values, alpha=0.1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(FEATURE_COLS, size=8)
ax.set_title("Linguistic Feature Profiles by Condition")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(SUMMARY_DIR / "fig4_radar_features.png", dpi=150)
plt.show()

### Figure 5: Feature correlation heatmap

In [ ]:
corr_cols = FEATURE_COLS + ["gptzero_human_prob"]
merged_ling = linguistic.merge(
    detection[["task_id", "condition", "run_id", "gptzero_human_prob"]],
    on=["task_id", "condition", "run_id"], how="inner",
)
corr_matrix = merged_ling[corr_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Feature Correlation with Detector Scores")
plt.tight_layout()
plt.savefig(SUMMARY_DIR / "fig5_feature_correlation.png", dpi=150)
plt.show()

### Figure 6: Detectability by model (H6)

In [ ]:
if "model" in detection.columns and detection["model"].nunique() > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, cond in zip(axes, ["context_rich", "style_constrained"]):
        subset = detection[detection["condition"] == cond]
        sns.barplot(data=subset, x="model", y="gptzero_human_prob", ax=ax, ci=95)
        ax.set_title(f"Detectability by Model ({cond})")
        ax.set_ylabel("GPTZero Human Probability")
        ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()
    plt.savefig(SUMMARY_DIR / "fig6_model_comparison.png", dpi=150)
    plt.show()

    # Kruskal-Wallis test
    for cond in ["context_rich", "style_constrained"]:
        groups = [g["gptzero_human_prob"].values
                  for _, g in detection[detection["condition"] == cond].groupby("model")]
        if len(groups) >= 2:
            stat, p = kruskal(*groups)
            print(f"H4 ({cond}): Kruskal-Wallis H={stat:.3f}, p={p:.4f}")
else:
    print("Model comparison: only one model found in detection results, skipping.")

### Figure 7: Detectability by content length (H7)

In [ ]:
if "word_target" in detection.columns and detection["word_target"].nunique() > 1:
    length_order = ["800-1000", "1500-2000"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, cond in zip(axes, ["context_rich", "style_constrained"]):
        subset = detection[detection["condition"] == cond]
        available = [l for l in length_order if l in subset["word_target"].values]
        sns.barplot(data=subset, x="word_target", y="gptzero_human_prob",
                    order=available, ax=ax, ci=95)
        ax.set_title(f"Detectability by Length ({cond})")
        ax.set_ylabel("GPTZero Human Probability")
        ax.set_xlabel("Word Target")
    plt.tight_layout()
    plt.savefig(SUMMARY_DIR / "fig7_length_variation.png", dpi=150)
    plt.show()

    # Spearman correlation: word count vs detectability
    all_texts_for_len = collect_all_texts()
    wc_map = {(t.task_id, t.condition, t.run_id): len(t.text.split())
              for t in all_texts_for_len}
    detection["word_count"] = detection.apply(
        lambda r: wc_map.get((r["task_id"], r["condition"], r["run_id"]), 0), axis=1)
    rho, p = spearmanr(detection["word_count"], detection["gptzero_human_prob"])
    print(f"H5: Spearman rho={rho:.3f}, p={p:.4f} (word count vs human_prob)")
else:
    print("Length variation: only one word_target found in detection results, skipping.")